# Desenvolvimento de um POS Tagger em Português Brasileiro com BERT-pt
### *POS Tagging for Brazilian Portuguese Using BERT-pt*

Desenvolvedora: Lorena Melo


## Introdução

O presente trabalho aborda a implementação e análise de um modelo de etiquetagem gramatical (POS Tagging) para o Português Brasileiro utilizando o modelo **BERT-pt**, uma variação do BERT pré-treinado para o idioma português. A tarefa de POS Tagging consiste em identificar e classificar cada palavra de um texto em sua categoria gramatical correspondente, como substantivo, verbo ou adjetivo. Essa tarefa é fundamental para diversas aplicações em Processamento de Linguagem Natural (PLN), incluindo tradutores automáticos, análise sintática e sistemas de resposta a perguntas.

O dataset utilizado foi o **MacMorpho**, uma base de dados robusta e amplamente utilizada para estudos em PLN no Português. Este trabalho visa explorar a eficácia de modelos baseados em transformadores, como o BERT-pt, para tarefas de etiquetagem gramatical, bem como identificar pontos fortes e áreas de melhoria.

## Metodologia

### Dados

O conjunto de dados utilizado foi o **MacMorpho**, dividido nos conjuntos de treino, validação e teste. Cada instância é composta por uma palavra e sua respectiva etiqueta gramatical.

### Modelo

Utilizou-se o **BERT-pt**, modelo baseado na arquitetura Transformer pré-treinado em grandes corpora de textos em português. Foi adaptado para classificação de tokens (*Token Classification*) adicionando uma camada linear para prever as etiquetas gramaticais.

### Treinamento

**Hiperparâmetros principais:**
- Taxa de aprendizado: **2e-5**.
- Tamanho do batch: **16**.
- Épocas: **3**.
- Comprimento máximo dos tokens: **128**.

**Métrica de avaliação:**
Utilizou-se o **F1-score** como principal métrica para avaliar o desempenho por classe gramatical, além de **precisão**, **recall** e **acurácia geral**.

### Implementação

O modelo foi implementado utilizando a biblioteca **Hugging Face Transformers**, que fornece ferramentas robustas para treinamento e avaliação de modelos baseados em transformadores.

**Fluxo principal:**
1. Tokenização dos dados com padding e truncamento.
2. Alinhamento das etiquetas com os tokens gerados pelo modelo.
3. Treinamento utilizando o `Trainer` da Hugging Face.
4. Avaliação com base no conjunto de teste.

In [1]:
import os
from transformers import AutoTokenizer
import pandas as pd

## Passo 1: Carregamento e Pré-processamento dos Dados

 ## 1. Carregamento dos Dados

In [2]:
# Caminhos dos arquivos
train_file = "data/macmorpho-train.txt"
dev_file = "data/macmorpho-dev.txt"
test_file = "data/macmorpho-test.txt"

In [3]:
# Função para carregar os dados
def carregar_dados_ajustado(file_path):
    sentences, tags = [], []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # Verifica se a linha não está vazia
                tokens_and_tags = line.split(" ")  # Divide a linha em tokens
                sentence = []
                tag = []
                for pair in tokens_and_tags:
                    if "_" in pair:  # Verifica se há um "_" separando token e tag
                        token, label = pair.rsplit("_", 1)  # Divide token e etiqueta
                        sentence.append(token)
                        tag.append(label)
                    else:
                        print(f"Formato inválido encontrado: {pair}")
                sentences.append(sentence)
                tags.append(tag)
    return sentences, tags

# Carregar os dados
train_sentences, train_tags = carregar_dados_ajustado(train_file)
dev_sentences, dev_tags = carregar_dados_ajustado(dev_file)
test_sentences, test_tags = carregar_dados_ajustado(test_file)




In [9]:
print(f"Treino: {len(train_sentences)} sentenças")
print(f"Validação: {len(dev_sentences)} sentenças")
print(f"Teste: {len(test_sentences)} sentenças")

Treino: 21955 sentenças
Validação: 1997 sentenças
Teste: 9987 sentenças


## 2. Tokenização e Alinhamento

Aqui teremos:

- **Tokenização das Sentenças:** Usaremos o tokenizer do BERT-pt para dividir as sentenças em tokens do modelo e alinhar as etiquetas correspondentes.

- **Alinhamento das Etiquetas:** Como o tokenizer pode dividir palavras em subpalavras, precisamos garantir que as etiquetas sejam ajustadas para corresponder aos tokens gerados.


In [4]:
from transformers import AutoTokenizer

# Carregar o tokenizer do BERT-pt
tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

def tokenizar_e_alinhar(sentences, tags, tokenizer, label_to_id, max_length=128):
    tokenized_inputs = []
    aligned_labels = []

    for sentence, label in zip(sentences, tags):
        # Tokenizar sentença
        tokenized = tokenizer(sentence, truncation=True, is_split_into_words=True, padding="max_length", max_length=max_length)
        word_ids = tokenized.word_ids()  # Mapeamento dos tokens para as palavras originais

        # Alinhar etiquetas com os tokens
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # Ignorar tokens especiais
            else:
                label_ids.append(label_to_id[label[word_idx]])

        tokenized_inputs.append(tokenized)
        aligned_labels.append(label_ids)

    return tokenized_inputs, aligned_labels

# Criar mapeamento de etiquetas para IDs
unique_tags = set(tag for tag_list in train_tags for tag in tag_list)
label_to_id = {tag: idx for idx, tag in enumerate(sorted(unique_tags))}
id_to_label = {idx: tag for tag, idx in label_to_id.items()}

# Tokenizar e alinhar os dados de treino
train_inputs, train_labels = tokenizar_e_alinhar(train_sentences, train_tags, tokenizer, label_to_id)
dev_inputs, dev_labels = tokenizar_e_alinhar(dev_sentences, dev_tags, tokenizer, label_to_id)
test_inputs, test_labels = tokenizar_e_alinhar(test_sentences, test_tags, tokenizer, label_to_id)

print("Tokenização e alinhamento concluídos!")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenização e alinhamento concluídos!


In [5]:
#train_inputs
#it is a list of dictionaries - checking the train_inputs data

### 2.1 Conferindo a tokenização

- Exibir Tokens e Etiquetas:

    Adicionei um código para imprimir os tokens gerados e suas respectivas etiquetas para algumas sentenças do conjunto de treinamento (3 primeiros exemplos)

In [6]:
# Conferir a tokenização de algumas sentenças
def conferir_tokenizacao(tokenizer, sentences, tags, label_to_id, num_exemplos=3):
    for i in range(num_exemplos):
        print(f"Sentença {i + 1}: {' '.join(sentences[i])}")
        print(f"Etiquetas Originais: {tags[i]}")

        # Tokenizar e alinhar
        tokenized = tokenizer(sentences[i], truncation=True, is_split_into_words=True)
        word_ids = tokenized.word_ids()
        tokens = tokenizer.convert_ids_to_tokens(tokenized["input_ids"])
        labels = []

        for word_idx in word_ids:
            if word_idx is None:
                labels.append("IGNORE")
            else:
                labels.append(tags[i][word_idx])

        print(f"Tokens: {tokens}")
        print(f"Etiquetas Alinhadas: {labels}")
        print("-" * 50)

# Conferir os primeiros 3 exemplos
conferir_tokenizacao(tokenizer, train_sentences, train_tags, label_to_id)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Sentença 1: Jersei atinge média de Cr$ 1,4 milhão na venda da Pinhal em São Paulo .
Etiquetas Originais: ['N', 'V', 'N', 'PREP', 'CUR', 'NUM', 'N', 'PREP+ART', 'N', 'PREP+ART', 'NPROP', 'PREP', 'NPROP', 'NPROP', 'PU']
Tokens: ['[CLS]', 'Jer', '##sei', 'atinge', 'média', 'de', 'Cr', '$', '1', ',', '4', 'milhão', 'na', 'venda', 'da', 'Pin', '##hal', 'em', 'São', 'Paulo', '.', '[SEP]']
Etiquetas Alinhadas: ['IGNORE', 'N', 'N', 'V', 'N', 'PREP', 'CUR', 'CUR', 'NUM', 'NUM', 'NUM', 'N', 'PREP+ART', 'N', 'PREP+ART', 'NPROP', 'NPROP', 'PREP', 'NPROP', 'NPROP', 'PU', 'IGNORE']
--------------------------------------------------
Sentença 2: Programe sua viagem à Exposição Nacional do Zebu , que começa dia 25 .
Etiquetas Originais: ['V', 'PROADJ', 'N', 'PREP+ART', 'NPROP', 'NPROP', 'NPROP', 'NPROP', 'PU', 'PRO-KS', 'V', 'N', 'N', 'PU']
Tokens: ['[CLS]', 'Pro', '##gram', '##e', 'sua', 'viagem', 'à', 'Exposição', 'Nacional', 'do', 'Ze', '##bu', ',', 'que', 'começa', 'dia', '25', '.', '[SEP]']
Etique

## Passo 2: Criar Dataset para Treinamento

Vamos organizar os dados no formato necessário para o modelo `transformers`. O modelo  `transformers` utiliza o formato de `Dataset` para manipular os dados. Aqui, iremos criar conjuntos de dados de treino, validação e teste compatíveis com o `Trainer`.



In [7]:
from torch.utils.data import Dataset
import torch

class POSDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs  # Lista de dicionários
        self.labels = labels  # Lista de sequências de etiquetas

    def __len__(self):
        return len(self.inputs)  # Retorna o número de exemplos no dataset

    def __getitem__(self, idx):
        # Retorna um único exemplo como dicionário
        item = {key: torch.tensor(val) for key, val in self.inputs[idx].items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        print(item)  # Adicione este print para verificar o formato
        return item



# Criar os datasets
train_dataset = POSDataset(train_inputs, train_labels)
dev_dataset = POSDataset(dev_inputs, dev_labels)
test_dataset = POSDataset(test_inputs, test_labels)



print("Dataset criado com sucesso!")


Dataset criado com sucesso!


## Passo 3: Configurar o Modelo

Usaremos o `AutoModelForTokenClassification` da biblioteca transformers para treinar o modelo com base no BERT-pt.

In [8]:
# Configurar o Modelo

from transformers import AutoModelForTokenClassification

# Carregar o modelo para classificação de tokens
model = AutoModelForTokenClassification.from_pretrained(
    "neuralmind/bert-base-portuguese-cased",
    num_labels=len(label_to_id)
)

# Exibir o modelo para verificar
print(model)


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(29794, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

## Passo 4: Configurar o Treinamento

Vamos configurar os argumentos de treinamento e inicializar o `Trainer`.

In [ ]:
#! pip install accelerate==0.26.0

In [9]:
import accelerate
accelerate.__version__

'1.2.1'

In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_total_limit=2,  # Limitar checkpoints salvos,
    report_to="none"  # Disable wandb integration
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,

)




/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-10-32459a4cde20>:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
# Testar um exemplo do dataset
print(train_dataset[0])  # Deve conter input_ids, attention_mask e labels com tamanhos consistentes

{'input_ids': tensor([  101,  4166, 15335, 11903,  2517,   125, 12064,   109,   205,   117,
          678,  6271,   229,  5304,   180, 13820,  5954,   173,   710,  1033,
          119,   102,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 

### 4.1 Treinando o Modelo


In [12]:
import os
os.environ["WANDB_DISABLED"] = "true" #desativar completamente o W&B do huggingface

# Iniciar o treinamento
trainer.train()

{'input_ids': tensor([ 101, 3348,  706, 8977,  107, 5121,  771,  107,  119,  102,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0

Epoch,Training Loss,Validation Loss
1,0.063900,0.068175
2,0.052400,0.064485
3,0.035800,0.064770


Streaming output truncated to the last 5000 lines.
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100])}
{'input_ids': tensor([  101,  1453,   730,   809,   256,   529,  3637,   171,   395, 22312,
          117,  1315,  1096,   320, 11572,   119,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
         

TrainOutput(global_step=7116, training_loss=0.08169302749308795, metrics={'train_runtime': 3228.1076, 'train_samples_per_second': 35.266, 'train_steps_per_second': 2.204, 'total_flos': 7438380641998848.0, 'train_loss': 0.08169302749308795, 'epoch': 3.0})

**Avaliação no Conjunto de Teste:**

Após o treinamento, avaliamos o modelo no conjunto de teste para obter métricas como **precisão, recall e F1-score:**

In [13]:
predictions, labels, _ = trainer.predict(test_dataset)

{'input_ids': tensor([ 101, 1920,  183, 1118,  102,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0

Streaming output truncated to the last 5000 lines.
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100])}
{'input_ids': tensor([  101,  1097,  9877,   118,   176,   131,   327,  2196,  1479,   117,
          230, 16014,   125, 16610, 12670,   285,   412,  3167,  8680,   125,
 

Com este código, agora temos as previsões do modelo (predictions) e os rótulos verdadeiros (labels) para o conjunto de teste.

👉 O próximo passo é calcular as métricas de avaliação, como **precisão**, **recall**, e **F1-score**, para analisar o desempenho do modelo.



## Resultados

### 1. Preparar as Previsões e Rótulos
As previsões retornadas pelo `trainer.predict` estão no formato de logits (valores contínuos). Precisamos converter esses logits em índices de classes para compará-los com os rótulos verdadeiros.

In [14]:
import numpy as np

# Converter logits em índices de classes
predictions = np.argmax(predictions, axis=2)  # A dimensão 2 contém as classes

# Remover valores ignorados (-100) para alinhamento
true_labels = []
true_predictions = []

for label, prediction in zip(labels, predictions):
    valid_indices = label != -100  # Ignorar tokens de padding
    true_labels.append(label[valid_indices])
    true_predictions.append(prediction[valid_indices])


### 2. Calcular Métricas
Usaremos `classification_report` da biblioteca sklearn para calcular as métricas.

In [15]:
from sklearn.metrics import classification_report

# Mapear IDs para etiquetas
label_names = [id_to_label[i] for i in range(len(id_to_label))]

# Concatenar todas as etiquetas e previsões em uma única lista
flat_labels = [item for sublist in true_labels for item in sublist]
flat_predictions = [item for sublist in true_predictions for item in sublist]

# Relatório de classificação
report = classification_report(flat_labels, flat_predictions, target_names=label_names)
print(report)


              precision    recall  f1-score   support

         ADJ       0.96      0.95      0.95     12333
         ADV       0.94      0.93      0.94      6100
      ADV-KS       0.86      0.86      0.86       232
         ART       0.99      0.99      0.99     12566
         CUR       0.99      1.00      0.99       592
          IN       0.50      0.77      0.61       146
          KC       0.99      0.98      0.98      4533
          KS       0.93      0.92      0.93      2538
           N       0.98      0.98      0.98     48485
       NPROP       0.98      0.98      0.98     26517
         NUM       0.98      0.96      0.97      3576
         PCP       0.97      0.96      0.96      5630
        PDEN       0.88      0.91      0.89      1130
        PREP       0.98      0.99      0.98     16776
    PREP+ADV       1.00      0.78      0.88        41
    PREP+ART       0.99      0.99      0.99     10211
 PREP+PRO-KS       0.87      0.93      0.90        58
 PREP+PROADJ       1.00    

### 3. Analisar as Métricas
O relatório gerado pelo classification_report inclui métricas como:

- Precisão: Proporção de predições corretas para cada classe.
- Recall: Proporção de exemplos verdadeiros identificados corretamente.
- F1-score: Média harmônica entre precisão e recall.

</br>

---

</br>

### Análise dos Resultados

1. **Classes com Maior Desempenho**

  As classes com os melhores valores de precisão, recall e F1-score indicam que o modelo está conseguindo identificar essas categorias com alta confiabilidade.

  **Melhores Classes Gramaticais** (Precision = 1.00):
  - PREP+PROADJ (F1: 1.00, Support: 317)
  - PU (F1: 1.00, Support: 27544)
  - PROPESS (F1: 0.99, Support: 3349)
  - CUR (F1: 0.99, Support: 592)
  - ART (F1: 0.99, Support: 12566)
  - V (F1: 0.99, Support: 26062)

  Essas classes possuem alta precisão e recall, indicando que o modelo raramente comete erros ao identificá-las. Isso se deve, em parte, ao fato de que essas classes são bem definidas e possuem características linguísticas distintas, como PU (pontuação) e ART (artigos), que têm formas fixas e claras.


2. **Classes com Menor Desempenho**

  As classes com menor precisão e F1-score apontam para dificuldades do modelo em distinguir essas categorias.

  **Classes com Menor Precisão**:
    - IN (Precision: 0.53, Recall: 0.79, F1: 0.63, Support: 146)
    - ADV-KS (Precision: 0.85, Recall: 0.86, F1: 0.86, Support: 232)
    - PREP+PRO-KS (Precision: 0.89, Recall: 0.93, F1: 0.91, Support: 58)

  A classe IN apresenta o menor desempenho geral, o que pode ser explicado pelo pequeno número de exemplos no conjunto de treinamento (Support: 146) e pela possível ambiguidade na categorização de itens dessa classe.
    
  As classes ADV-KS e PREP+PRO-KS têm desempenho médio, o que sugere que o modelo enfrenta dificuldades em identificar combinações específicas de advérbios ou preposições.

3. Análise Geral

  - Acurácia Geral: 98%, indicando que o modelo classifica corretamente a maioria dos tokens.

  - Macro Média:
    - Precision: 94%
    - Recall: 94%
    - F1-score: 94%
  Isso reflete um bom desempenho geral, mesmo considerando as classes menos representadas.

  - Weighted Média:
    - Precision: 98%
    - Recall: 98%
    - F1-score: 98%
  Isso demonstra que o modelo dá mais peso às classes mais frequentes, como N (substantivos) e PU (pontuação).

## Resumo Geral do Desempenho

In [25]:
# Exibir em formato tabular
print("Resumo Geral do Desempenho")
print("-" * 50)  # Linha separadora
print(f"{'Métrica':<15}{'Precision':<10}{'Recall':<10}{'F1':<10}{'Support':<10}")
print("-" * 50)  # Linha separadora
print(f"{'Accuracy':<15}{'-':<10}{'-':<10}{accuracy:.4f}{len(flat_labels):>10}")
print("-" * 50)  # Linha separadora
print(f"{'Macro Avg':<15}{macro_avg['precision']:<10.4f}{macro_avg['recall']:<10.4f}{macro_avg['f1-score']:<10.4f}{'-':<10}")
print("-" * 50)  # Linha separadora
print(f"{'Weighted Avg':<15}{weighted_avg['precision']:<10.4f}{weighted_avg['recall']:<10.4f}{weighted_avg['f1-score']:<10.4f}{'-':<10}")


Resumo Geral do Desempenho
--------------------------------------------------
Métrica        Precision Recall    F1        Support   
--------------------------------------------------
Accuracy       -         -         0.9792    216329
--------------------------------------------------
Macro Avg      0.9421    0.9429    0.9407    -         
--------------------------------------------------
Weighted Avg   0.9794    0.9792    0.9793    -         


In [28]:
import json

# Salvar o relatório em formato JSON no arquivo de texto
with open("classification_report.txt", "w") as f:
    f.write(json.dumps(report, indent=4))



### Pontos Consideráveis no Trabalho atual:

#### Pontos Fortes

O modelo teve excelente desempenho em classes gramaticais bem definidas e frequentes, como PU (pontuação), ART (artigos), e V (verbos).
Classes com características distintas e menos ambiguidade apresentaram alta precisão e recall.


#### Pontos Fracos

Classes menos frequentes no conjunto de treinamento, como IN (interjeições), tiveram baixo desempenho. Isso indica que o modelo pode beneficiar-se de mais exemplos dessas classes.
Categorias combinadas, como PREP+PRO-KS, também apresentam dificuldades, possivelmente devido à complexidade contextual.

## Discussão




### Discussão: Comparação com Trabalhos Anteriores

#### **Trabalho Anterior: "NLP Portuguese POS Tagger"**
- **Autora:** Lisa Terumi Oda ([GitHub Repository](https://github.com/lisaterumi/nlp-portuguese-postagger))
- **Modelo Utilizado:** O trabalho utilizou o modelo **BERTimbau**, um modelo pré-treinado exclusivamente para o idioma português, adaptado para a tarefa de POS Tagging.
- **Conjunto de Dados:** Assim como o presente trabalho, o **MacMorpho** foi utilizado como base.
- **Resultados:**  
  - Acurácia: **98.26%**  
  - F1-Score (Macro): **95%**  
  - F1-Score (Weighted): **98%**  

#### **Comparação com Este Trabalho**
- **Modelo Utilizado:** Este trabalho empregou o **BERT-pt**, outro modelo baseado em BERT pré-treinado para o português. Embora semelhante ao BERTimbau, os dois modelos possuem diferenças em suas arquiteturas e dados de treinamento.
- **Conjunto de Dados:** Ambos os trabalhos utilizaram o mesmo conjunto de dados, garantindo uma comparação justa.
- **Resultados:**  
  - Acurácia: **98%**  
  - F1-Score (Macro): **94%**  
  - F1-Score (Weighted): **98%**  

#### **Análise Comparativa**
1. **Desempenho Geral:**  
   - Ambos os trabalhos atingiram resultados muito próximos, com **acurácia geral de 98%**.
   - O F1-score ponderado também foi equivalente (**98%**), refletindo alta confiabilidade em classes mais frequentes, como `PU` (Pontuação) e `V` (Verbos).

2. **Desempenho em Classes Menos Representadas:**  
   - Este trabalho apresentou um F1-score Macro ligeiramente inferior (**94%** vs. **95%**), indicando que o modelo enfrentou mais dificuldades em classes com baixo suporte, como `IN` (Interjeição) e `ADV-KS` (Advérbio Subordinativo).

3. **Contribuições Diferenciadas:**  
   - O trabalho da Lisa demonstrou a eficácia do **BERTimbau** em tarefas de etiquetagem gramatical, enquanto este trabalho destacou a viabilidade de utilizar o **BERT-pt** como alternativa competitiva.

#### **Contribuições do Presente Trabalho**
- **Metodologia Reprodutível:** Os resultados alcançados com o **BERT-pt** mostram que diferentes modelos baseados em BERT podem alcançar desempenhos semelhantes, ampliando as opções para tarefas de POS Tagging no português.
- **Desempenho em Classes Compostas:** Este trabalho apresentou bons resultados em classes como `PREP+PROPESS` e `PREP+PROADJ`, sugerindo robustez em contextos mais complexos.

#### **Citação**
Lisa Terumi Oda. [NLP Portuguese POS Tagger](https://github.com/lisaterumi/nlp-portuguese-postagger). Disponível em: [https://github.com/lisaterumi/nlp-portuguese-postagger](https://github.com/lisaterumi/nlp-portuguese-postagger).
